## 🎵 Compose Music from Text with Claude Sonnet + Python

This notebook lets you generate music using a plain English text prompt. Claude Sonnet (via API) will return music notation in ABC format. Then, we’ll clean it up and convert it into a MIDI file using Python.

You can drag that MIDI file into any DAW or notation software.


---

**How it works:**
1. Authenticate with your Claude API key (securely)
2. Enter a music prompt (like “Write a chord progression in C major”)
3. Claude replies in ABC notation
4. The script validates and converts it to a MIDI file

### Step 1: Set Up your Anthropic API Key
The cell below will prompt you to enter your API key securely. You'll need access to [Anthropic's console](https://console.anthropic.com/login?returnTo=%2F%3F) and an [API key](https://console.anthropic.com/login?selectAccount=true&returnTo=%2Fsettings%2Fkeys%3F).

In [1]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

KeyboardInterrupt: Interrupted by user

### Step 2: Call Claude Sonnet with a Music Prompt

In [ ]:
!pip install anthropic
import anthropic
import os

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

message = client.messages.create(
    model="claude-3-sonnet-20250219",
    max_tokens=1000,
    temperature=0.5,
    system="You are a music composer. Write only ABC notation. Use headers like T:, M:, L:, K:. Use square brackets for chords like [C E G]. Do not include any extra text.",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Write a chord progression in F major in 4/4"
                }
            ]
        }
    ]
)

raw_output = message.content[0].text
print(raw_output)

### Step 3: Clean and Validate ABC Notation
This prepares Claude’s output for Music21 and MIDI export.

In [ ]:
!pip install music21
from music21 import converter

def convert_sharps_flats(progression):
    note_letters = 'ABCDEFG'
    converted_progression = ""
    i = 0
    while i < len(progression):
        char = progression[i]
        if char in note_letters:
            if i+1 < len(progression) and progression[i+1] in ['#', 'b', '♭']:
                converted_char = '^' if progression[i+1] == '#' else '_'
                converted_progression += converted_char + char
                i += 1
            else:
                converted_progression += char
        else:
            converted_progression += char
        i += 1
    return converted_progression

def validate_abc_headers(text):
    return all(header in text for header in ["T:", "M:", "L:", "K:"])

def parse_abc_to_stream(abc_notation):
    try:
        return converter.parseData(abc_notation, format='abc')
    except Exception as e:
        print(f"Parse error: {e}")
        return None

def convert_stream_to_midi(s, file_name="output.mid"):
    s.write('midi', fp=file_name)

### 🎼 Step 4: Convert to MIDI

In [ ]:
cleaned = convert_sharps_flats(raw_output)

if validate_abc_headers(cleaned):
    stream = parse_abc_to_stream(cleaned)
    if stream:
        convert_stream_to_midi(stream)
        print("✅ Saved as output.mid")
    else:
        print("Could not parse ABC notation")
else:
    print("Missing required ABC headers")

### Play your MIDI file! 🎧

In [ ]:
from IPython.display import Audio
Audio("output.mid")